# Clean LSTM trainer for `cont_data_labeled2.npz`

This notebook is intentionally minimal. It does only the following:

1. Loads `cont_data_labeled2.npz`.
2. Uses the sample-level labels already stored in the file.
3. Creates a chronological 90/10 train/validation split.
4. Builds sliding-window LSTM inputs from either FFT band power or raw signal, controlled by `FEATURE_MODE`.
5. Trains one LSTM classifier.
6. Saves a reusable checkpoint and includes a loader/prediction utility.

Label convention:

- `0 = Idle`
- `1 = Beta`


In [ ]:
# Optional Google Drive mount for Colab.
from pathlib import Path

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('Google Drive already mounted.')
except Exception as exc:
    print(f'Google Drive mount skipped: {exc}')


## Imports and user-editable configuration

Edit `FEATURE_MODE` to switch the model input:

- `"fft_bandpower"`: one log FFT band-power feature per window.
- `"raw_signal"`: the full raw waveform window as the LSTM sequence.


In [ ]:
import json
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# =============================================================================
# USER-EDITABLE SETTINGS
# =============================================================================

# Change this flag to choose the LSTM input representation.
FEATURE_MODE = 'fft_bandpower'   # options: 'fft_bandpower' or 'raw_signal'

# Main labeled data file produced by sgv4_relabel_fixed_threshold.ipynb.
DATA_NPZ = Path('/content/drive/MyDrive/1 - BME PhD/BCI Lab/online classification/signal_generator/Drive Code/cont_data_labeled2.npz')

# Fallback for the earlier non-"Drive Code" location, if that is where the file exists.
ALT_DATA_NPZ = Path('/content/drive/MyDrive/1 - BME PhD/BCI Lab/online classification/signal_generator/cont_data_labeled2.npz')

# Where the trained model checkpoint and small metadata files will be saved.
OUTPUT_DIR = DATA_NPZ.parent / 'sg_results_clean_lstm'


@dataclass
class Config:
    data_npz: str = str(DATA_NPZ)
    alt_data_npz: str = str(ALT_DATA_NPZ)
    output_dir: str = str(OUTPUT_DIR)

    feature_mode: str = FEATURE_MODE
    train_fraction: float = 0.90

    # Sliding windows.
    window_sec: float = 1.0
    stride_sec: float = 0.25
    label_mode: str = 'endpoint'   # options: 'endpoint' or 'majority'

    # Used only when feature_mode == 'fft_bandpower'.
    bandpower_hz: tuple = (18.0, 22.0)

    # LSTM training.
    hidden_size: int = 4
    num_layers: int = 1
    dropout: float = 0.0
    batch_size: int = 64
    epochs: int = 12
    lr: float = 1e-3
    seed: int = 888

    # Binary classification: 0=Idle, 1=Beta.
    num_classes: int = 2


cfg = Config()

VALID_FEATURE_MODES = {'fft_bandpower', 'raw_signal'}
VALID_LABEL_MODES = {'endpoint', 'majority'}

if cfg.feature_mode not in VALID_FEATURE_MODES:
    raise ValueError(f'feature_mode must be one of {sorted(VALID_FEATURE_MODES)}, got {cfg.feature_mode!r}')
if cfg.label_mode not in VALID_LABEL_MODES:
    raise ValueError(f'label_mode must be one of {sorted(VALID_LABEL_MODES)}, got {cfg.label_mode!r}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'FEATURE_MODE = {cfg.feature_mode}')


## Reproducibility and path helpers

In [ ]:
def set_seed(seed: int) -> None:
    """Set Python, NumPy, and PyTorch RNG seeds."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Keep this lightweight; strict deterministic algorithms can break some CUDA LSTM paths.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False


def resolve_data_path(primary_path, alt_path=None) -> Path:
    """Resolve the labeled npz file across Colab Drive and local inspection environments."""
    primary_path = Path(primary_path)
    candidates = [primary_path]

    if alt_path is not None:
        candidates.append(Path(alt_path))

    # Local fallbacks used when inspecting/running outside Colab.
    candidates.extend([
        Path.cwd() / primary_path.name,
        Path('/mnt/data') / primary_path.name,
    ])

    # Deduplicate while preserving order.
    seen = set()
    unique_candidates = []
    for path in candidates:
        key = str(path)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(path)

    for path in unique_candidates:
        if path.exists():
            return path

    msg = 'Could not find cont_data_labeled2.npz. Checked:\n' + '\n'.join(f'  - {p}' for p in unique_candidates)
    raise FileNotFoundError(msg)


def resolve_output_dir(output_dir) -> Path:
    """Create the output directory; fall back locally if the configured path is unavailable."""
    output_dir = Path(output_dir)
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        return output_dir
    except Exception as exc:
        fallback = Path('/mnt/data/sg_results_clean_lstm') if Path('/mnt/data').exists() else Path.cwd() / 'sg_results_clean_lstm'
        fallback.mkdir(parents=True, exist_ok=True)
        print(f'Configured output directory unavailable: {output_dir}')
        print(f'Reason: {exc}')
        print(f'Using fallback output directory: {fallback}')
        return fallback


set_seed(cfg.seed)


## Load `cont_data_labeled2.npz`

Required keys:

- `eeg`
- `samplerate`
- `sample_labels`


In [ ]:
def load_labeled_npz(cfg: Config):
    """Load a single-channel labeled continuous signal."""
    data_path = resolve_data_path(cfg.data_npz, cfg.alt_data_npz)
    labeled = np.load(data_path)

    required = {'eeg', 'samplerate', 'sample_labels'}
    missing = required.difference(set(labeled.files))
    if missing:
        raise KeyError(f'Labeled file is missing required key(s): {sorted(missing)}. Found keys: {labeled.files}')

    eeg = np.asarray(labeled['eeg'], dtype=np.float32).squeeze()
    fs = int(np.asarray(labeled['samplerate']).item())
    sample_labels = np.asarray(labeled['sample_labels'], dtype=np.int64).reshape(-1)

    # This cleaned notebook is intentionally single-channel.
    if eeg.ndim == 2:
        if min(eeg.shape) == 1:
            eeg = eeg.reshape(-1)
        elif eeg.shape[0] < eeg.shape[1]:
            print(f'EEG has shape {eeg.shape}; using first row as the single channel.')
            eeg = eeg[0, :]
        else:
            print(f'EEG has shape {eeg.shape}; using first column as the single channel.')
            eeg = eeg[:, 0]
    elif eeg.ndim != 1:
        raise ValueError(f'Expected 1D EEG or 2D single-channel EEG, got shape {eeg.shape}')

    n = min(len(eeg), len(sample_labels))
    eeg = eeg[:n].astype(np.float32)
    sample_labels = sample_labels[:n].astype(np.int64)

    unique_labels = set(np.unique(sample_labels).tolist())
    if not unique_labels.issubset({0, 1}):
        raise ValueError(f'sample_labels must contain only 0 and 1. Found: {sorted(unique_labels)}')

    counts = dict(zip(*np.unique(sample_labels, return_counts=True)))
    print(f'Loaded: {data_path}')
    print(f'Samples: {n:,}')
    print(f'Samplerate: {fs} Hz')
    print(f'Duration: {n / fs:.2f} s')
    print(f'Label counts: {counts}')
    print('Label convention: 0=Idle, 1=Beta')

    return eeg, sample_labels, fs, data_path


eeg, sample_labels, fs, data_path = load_labeled_npz(cfg)
output_dir = resolve_output_dir(cfg.output_dir)


## Feature extraction and chronological 90/10 split

In [ ]:
def split_continuous_signal(eeg, sample_labels, train_fraction: float):
    """Split the continuous sample stream chronologically."""
    n = min(len(eeg), len(sample_labels))
    split_sample = int(round(n * float(train_fraction)))
    split_sample = max(1, min(n - 1, split_sample))

    train_eeg = eeg[:split_sample]
    train_labels = sample_labels[:split_sample]
    val_eeg = eeg[split_sample:]
    val_labels = sample_labels[split_sample:]

    return train_eeg, train_labels, val_eeg, val_labels, split_sample


def window_label(labels_window, mode='endpoint') -> int:
    """Assign one class label to a sliding window."""
    labels_window = np.asarray(labels_window, dtype=np.int64)
    if mode == 'endpoint':
        return int(labels_window[-1])
    if mode == 'majority':
        counts = np.bincount(labels_window, minlength=2)
        return int(np.argmax(counts))
    raise ValueError("label_mode must be 'endpoint' or 'majority'.")


def fft_log_bandpower_1d(x, fs: int, band=(18.0, 22.0), eps=1e-12) -> float:
    """Return log10 FFT band power for one 1D window."""
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    if len(x) < 2:
        raise ValueError('FFT band power requires at least two samples per window.')

    x = x - np.mean(x)
    taper = np.hanning(len(x)).astype(np.float32)
    xw = x * taper

    freqs = np.fft.rfftfreq(len(xw), d=1.0 / fs)
    fft_vals = np.fft.rfft(xw)
    power = np.abs(fft_vals) ** 2

    low, high = float(band[0]), float(band[1])
    mask = (freqs >= low) & (freqs <= high)
    if not np.any(mask):
        raise ValueError(f'No FFT bins inside band={band} for fs={fs} and window length={len(x)}.')

    return float(np.log10(np.sum(power[mask]) + eps))


def extract_features_from_signal(eeg, fs: int, cfg: Config):
    """Create LSTM input features from one continuous signal without needing labels."""
    eeg = np.asarray(eeg, dtype=np.float32).reshape(-1)

    win = int(round(float(cfg.window_sec) * fs))
    stride = int(round(float(cfg.stride_sec) * fs))
    if win <= 0 or stride <= 0:
        raise ValueError('window_sec and stride_sec must produce positive sample counts.')
    if len(eeg) < win:
        raise ValueError(f'Not enough samples ({len(eeg)}) for window length ({win}).')

    X, start_samples, end_samples = [], [], []
    for start in range(0, len(eeg) - win + 1, stride):
        end = start + win
        raw_window = eeg[start:end]

        if cfg.feature_mode == 'raw_signal':
            # Shape per window: (time, features) = (window_samples, 1)
            features = raw_window[:, None]
        elif cfg.feature_mode == 'fft_bandpower':
            # Shape per window: (time, features) = (1, 1)
            bp = fft_log_bandpower_1d(raw_window, fs=fs, band=cfg.bandpower_hz)
            features = np.asarray([[bp]], dtype=np.float32)
        else:
            raise ValueError(f'Unknown feature_mode: {cfg.feature_mode!r}')

        X.append(features.astype(np.float32))
        start_samples.append(start)
        end_samples.append(end)

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(start_samples, dtype=np.int64),
        np.asarray(end_samples, dtype=np.int64),
    )


def make_labeled_windows(eeg, sample_labels, fs: int, cfg: Config):
    """Create LSTM windows and one class label per window."""
    eeg = np.asarray(eeg, dtype=np.float32).reshape(-1)
    sample_labels = np.asarray(sample_labels, dtype=np.int64).reshape(-1)
    n = min(len(eeg), len(sample_labels))
    eeg = eeg[:n]
    sample_labels = sample_labels[:n]

    X, start_samples, end_samples = extract_features_from_signal(eeg, fs, cfg)
    y = np.asarray([
        window_label(sample_labels[start:end], mode=cfg.label_mode)
        for start, end in zip(start_samples, end_samples)
    ], dtype=np.int64)

    return X, y, start_samples, end_samples


def fit_normalizer(X_train):
    """Fit a feature-wise standardizer using training windows only."""
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def apply_normalizer(X, mean, std):
    return ((X - mean) / std).astype(np.float32)


In [ ]:
train_eeg, train_labels, val_eeg, val_labels, split_sample = split_continuous_signal(
    eeg,
    sample_labels,
    train_fraction=cfg.train_fraction,
)

X_train, y_train, train_starts, train_ends = make_labeled_windows(train_eeg, train_labels, fs, cfg)
X_val, y_val, val_starts, val_ends = make_labeled_windows(val_eeg, val_labels, fs, cfg)

normalizer_mean, normalizer_std = fit_normalizer(X_train)
X_train_norm = apply_normalizer(X_train, normalizer_mean, normalizer_std)
X_val_norm = apply_normalizer(X_val, normalizer_mean, normalizer_std)

print(f'Chronological split sample: {split_sample:,} / {len(eeg):,}')
print(f'Train samples: {len(train_eeg):,}; validation samples: {len(val_eeg):,}')
print(f'Window length: {cfg.window_sec} s = {int(round(cfg.window_sec * fs))} samples')
print(f'Stride: {cfg.stride_sec} s = {int(round(cfg.stride_sec * fs))} samples')
print(f'Feature mode: {cfg.feature_mode}')
print(f'X_train shape: {X_train_norm.shape}; y_train shape: {y_train.shape}')
print(f'X_val shape:   {X_val_norm.shape}; y_val shape:   {y_val.shape}')
print(f'Train label counts: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Val label counts:   {dict(zip(*np.unique(y_val, return_counts=True)))}')


## Dataset and LSTM architecture

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.long)

    def __len__(self):
        return int(len(self.y))

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class LSTMClassifier(nn.Module):
    """Minimal LSTM classifier for binary Idle/Beta classification."""
    def __init__(self, input_size: int, hidden_size: int, num_layers: int, num_classes: int = 2, dropout: float = 0.0):
        super().__init__()
        effective_dropout = float(dropout) if int(num_layers) > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=int(input_size),
            hidden_size=int(hidden_size),
            num_layers=int(num_layers),
            batch_first=True,
            dropout=effective_dropout,
        )
        self.fc = nn.Linear(int(hidden_size), int(num_classes))

    def forward(self, x):
        # x shape: (batch, time, features)
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        logits = self.fc(last_hidden)
        return logits


## Training and validation utilities

In [ ]:
def evaluate_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    all_true, all_pred, all_prob = [], [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)

            total_loss += float(loss.item()) * len(yb)
            total_correct += int((preds == yb).sum().item())
            total += len(yb)
            all_true.append(yb.cpu().numpy())
            all_pred.append(preds.cpu().numpy())
            all_prob.append(probs.cpu().numpy())

    return {
        'loss': total_loss / max(total, 1),
        'acc': total_correct / max(total, 1),
        'true': np.concatenate(all_true) if all_true else np.array([], dtype=np.int64),
        'pred': np.concatenate(all_pred) if all_pred else np.array([], dtype=np.int64),
        'prob': np.concatenate(all_prob) if all_prob else np.empty((0, 2), dtype=np.float32),
    }


def train_lstm_model(X_train, y_train, X_val, y_val, cfg: Config, device):
    set_seed(cfg.seed)

    train_dataset = WindowDataset(X_train, y_train)
    val_dataset = WindowDataset(X_val, y_val)

    generator = torch.Generator()
    generator.manual_seed(cfg.seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    model = LSTMClassifier(
        input_size=X_train.shape[-1],
        hidden_size=cfg.hidden_size,
        num_layers=cfg.num_layers,
        num_classes=cfg.num_classes,
        dropout=cfg.dropout,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    history = []
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            total_loss += float(loss.item()) * len(yb)
            total_correct += int((torch.argmax(logits, dim=1) == yb).sum().item())
            total += len(yb)

        train_loss = total_loss / max(total, 1)
        train_acc = total_correct / max(total, 1)
        val_metrics = evaluate_model(model, val_loader, criterion, device)

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['acc'],
        }
        history.append(row)
        print(
            f"Epoch {epoch:03d}/{cfg.epochs:03d} | "
            f"train loss {train_loss:.4f}, acc {train_acc:.4f} | "
            f"val loss {val_metrics['loss']:.4f}, acc {val_metrics['acc']:.4f}"
        )

    history_df = pd.DataFrame(history)
    final_val_metrics = evaluate_model(model, val_loader, criterion, device)
    return model, history_df, final_val_metrics


## Train the LSTM

In [ ]:
model, history_df, val_metrics = train_lstm_model(
    X_train_norm,
    y_train,
    X_val_norm,
    y_val,
    cfg,
    DEVICE,
)

print('\nFinal validation accuracy:', f"{val_metrics['acc']:.4f}")
history_df


In [ ]:
# Optional compact training curve.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_df['epoch'], history_df['train_acc'], marker='o', label='train_acc')
ax.plot(history_df['epoch'], history_df['val_acc'], marker='o', label='val_acc')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title(f'LSTM training curve ({cfg.feature_mode})')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## Save checkpoint for later use

In [ ]:
def _json_safe(value):
    """Convert common NumPy/Path values into JSON-safe Python objects."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    return value


def save_checkpoint(model, cfg: Config, checkpoint_path: Path, extra_metadata: dict):
    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    model_config = {
        'input_size': int(extra_metadata['input_size']),
        'hidden_size': int(cfg.hidden_size),
        'num_layers': int(cfg.num_layers),
        'num_classes': int(cfg.num_classes),
        'dropout': float(cfg.dropout),
    }

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'model_config': model_config,
        'config': asdict(cfg),
        'normalizer_mean': normalizer_mean,
        'normalizer_std': normalizer_std,
        'fs': int(extra_metadata['fs']),
        'window_samples': int(extra_metadata['window_samples']),
        'stride_samples': int(extra_metadata['stride_samples']),
        'feature_mode': cfg.feature_mode,
        'bandpower_hz': tuple(float(x) for x in cfg.bandpower_hz),
        'label_mode': cfg.label_mode,
        'class_names': {0: 'Idle', 1: 'Beta'},
        'label_convention': '0=Idle, 1=Beta',
        'source_data_path': str(extra_metadata['source_data_path']),
        'split_sample': int(extra_metadata['split_sample']),
        'history': extra_metadata['history'],
        'final_val_acc': float(extra_metadata['final_val_acc']),
    }

    torch.save(checkpoint, checkpoint_path)
    return checkpoint_path, checkpoint


checkpoint_path = output_dir / f'lstm_{cfg.feature_mode}_checkpoint.pt'
metadata = {
    'input_size': X_train_norm.shape[-1],
    'fs': fs,
    'window_samples': int(round(cfg.window_sec * fs)),
    'stride_samples': int(round(cfg.stride_sec * fs)),
    'source_data_path': data_path,
    'split_sample': split_sample,
    'history': history_df.to_dict(orient='records'),
    'final_val_acc': val_metrics['acc'],
}

checkpoint_path, checkpoint = save_checkpoint(model, cfg, checkpoint_path, metadata)

history_csv = output_dir / f'lstm_{cfg.feature_mode}_history.csv'
history_df.to_csv(history_csv, index=False)

metadata_json = output_dir / f'lstm_{cfg.feature_mode}_metadata.json'
with open(metadata_json, 'w') as f:
    json.dump(_json_safe({k: v for k, v in checkpoint.items() if k != 'model_state_dict'}), f, indent=2)

print(f'Saved checkpoint: {checkpoint_path}')
print(f'Saved history:    {history_csv}')
print(f'Saved metadata:   {metadata_json}')


## Load the saved model and use it later

The functions below recreate the LSTM architecture, load the saved state dictionary, reapply the training normalizer, and return window-level predictions from a continuous signal.


In [ ]:
def load_saved_model(checkpoint_path, device=None):
    checkpoint_path = Path(checkpoint_path)
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    try:
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location=device)

    model_cfg = checkpoint['model_config']
    model = LSTMClassifier(**model_cfg)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    return model, checkpoint


def cfg_from_checkpoint(checkpoint) -> Config:
    saved = checkpoint['config'].copy()
    saved['feature_mode'] = checkpoint.get('feature_mode', saved.get('feature_mode', 'fft_bandpower'))
    saved['bandpower_hz'] = tuple(checkpoint.get('bandpower_hz', saved.get('bandpower_hz', (18.0, 22.0))))
    return Config(**saved)


@torch.no_grad()
def predict_from_continuous_signal(model, checkpoint, eeg_signal, fs=None, batch_size=256, device=None):
    """Return window-level probabilities and predictions for one continuous 1D signal."""
    if device is None:
        device = next(model.parameters()).device
    if fs is None:
        fs = int(checkpoint['fs'])
    if int(fs) != int(checkpoint['fs']):
        raise ValueError(f'fs mismatch: got {fs}, checkpoint expects {checkpoint["fs"]}')

    pred_cfg = cfg_from_checkpoint(checkpoint)
    X, starts, ends = extract_features_from_signal(eeg_signal, int(fs), pred_cfg)
    X_norm = apply_normalizer(X, checkpoint['normalizer_mean'], checkpoint['normalizer_std'])

    loader = DataLoader(
        torch.as_tensor(X_norm, dtype=torch.float32),
        batch_size=batch_size,
        shuffle=False,
    )

    probs_all = []
    for xb in loader:
        xb = xb.to(device)
        logits = model(xb)
        probs = torch.softmax(logits, dim=1)
        probs_all.append(probs.cpu().numpy())

    probs_all = np.concatenate(probs_all, axis=0)
    preds = np.argmax(probs_all, axis=1).astype(np.int64)

    return pd.DataFrame({
        'start_sample': starts,
        'end_sample': ends,
        'end_time_sec': ends / int(fs),
        'pred_label': preds,
        'pred_name': np.where(preds == 0, 'Idle', 'Beta'),
        'prob_idle': probs_all[:, 0],
        'prob_beta': probs_all[:, 1],
    })


# Smoke test: reload the saved checkpoint and predict on the validation segment.
loaded_model, loaded_checkpoint = load_saved_model(checkpoint_path, DEVICE)
val_prediction_df = predict_from_continuous_signal(
    loaded_model,
    loaded_checkpoint,
    val_eeg,
    fs=fs,
    batch_size=cfg.batch_size,
    device=DEVICE,
)

print(f'Reloaded model from: {checkpoint_path}')
val_prediction_df.head()


In [ ]:
# Optional: save validation window predictions from the reloaded model.
val_predictions_csv = output_dir / f'lstm_{cfg.feature_mode}_val_predictions.csv'
val_prediction_df.to_csv(val_predictions_csv, index=False)
print(f'Saved validation predictions: {val_predictions_csv}')


## Overlay raw-signal LSTM validation predictions with true labels and raw signal

This cell reads `lstm_raw_signal_val_predictions.csv`, aligns each window-level prediction to its validation-window endpoint, and overlays it with the validation raw signal and sample-level true labels.

Run the notebook with `FEATURE_MODE = 'raw_signal'` first so the CSV exists. The prediction timestamps are relative to the validation segment, matching `val_eeg` and `val_labels`.


In [ ]:
def plot_validation_prediction_overlay(
    predictions_csv,
    raw_signal,
    true_sample_labels,
    fs: int,
    title='Raw-signal LSTM validation predictions overlaid with true labels and raw signal',
    max_duration_sec=None,
    start_time_sec=0.0,
    save_path=None,
):
    """Overlay validation-window predictions, true sample labels, and the raw signal.

    Expected prediction CSV columns:
        start_sample, end_sample, end_time_sec, pred_label, prob_idle, prob_beta

    `start_sample`, `end_sample`, and `end_time_sec` are expected to be relative
    to the validation segment, because `predict_from_continuous_signal()` is run
    on `val_eeg` in this notebook.
    """
    predictions_csv = Path(predictions_csv)
    if not predictions_csv.exists():
        raise FileNotFoundError(
            f'Prediction CSV not found: {predictions_csv}\n'
            "Set FEATURE_MODE = 'raw_signal', rerun the training/prediction cells, "
            "and confirm that lstm_raw_signal_val_predictions.csv was saved."
        )

    pred_df = pd.read_csv(predictions_csv)
    required_cols = {'end_sample', 'pred_label'}
    missing_cols = required_cols.difference(pred_df.columns)
    if missing_cols:
        raise KeyError(f'{predictions_csv.name} is missing required column(s): {sorted(missing_cols)}')

    raw_signal = np.asarray(raw_signal, dtype=np.float32).reshape(-1)
    true_sample_labels = np.asarray(true_sample_labels, dtype=np.int64).reshape(-1)
    n = min(len(raw_signal), len(true_sample_labels))
    raw_signal = raw_signal[:n]
    true_sample_labels = true_sample_labels[:n]

    fs = int(fs)
    t_signal = np.arange(n, dtype=np.float64) / fs

    start_time_sec = float(start_time_sec)
    if max_duration_sec is None:
        end_time_sec = t_signal[-1] if n else 0.0
    else:
        end_time_sec = start_time_sec + float(max_duration_sec)

    signal_mask = (t_signal >= start_time_sec) & (t_signal <= end_time_sec)
    if not np.any(signal_mask):
        raise ValueError('Selected plotting interval contains no validation samples.')

    pred_df = pred_df.copy()
    pred_df['end_sample'] = pred_df['end_sample'].astype(int)
    pred_df['pred_label'] = pred_df['pred_label'].astype(int)
    pred_df['plot_time_sec'] = pred_df['end_sample'] / fs
    pred_df = pred_df[
        (pred_df['plot_time_sec'] >= start_time_sec)
        & (pred_df['plot_time_sec'] <= end_time_sec)
    ]

    fig, ax_signal = plt.subplots(figsize=(14, 5))
    ax_signal.plot(
        t_signal[signal_mask],
        raw_signal[signal_mask],
        linewidth=0.8,
        label='raw signal',
    )
    ax_signal.set_xlabel('Validation time (s)')
    ax_signal.set_ylabel('Raw signal amplitude')
    ax_signal.grid(True, alpha=0.3)

    ax_label = ax_signal.twinx()
    ax_label.step(
        t_signal[signal_mask],
        true_sample_labels[signal_mask],
        where='post',
        linewidth=1.4,
        alpha=0.85,
        label='true label',
    )

    if len(pred_df) > 0:
        ax_label.step(
            pred_df['plot_time_sec'].to_numpy(),
            pred_df['pred_label'].to_numpy(),
            where='post',
            linewidth=1.4,
            linestyle='--',
            alpha=0.95,
            label='predicted label',
        )
    else:
        print('No prediction rows fall inside the selected plotting interval.')

    ax_label.set_ylabel('Label: 0=Idle, 1=Beta')
    ax_label.set_yticks([0, 1])
    ax_label.set_ylim(-0.15, 1.15)

    handles_1, labels_1 = ax_signal.get_legend_handles_labels()
    handles_2, labels_2 = ax_label.get_legend_handles_labels()
    ax_signal.legend(handles_1 + handles_2, labels_1 + labels_2, loc='upper right')

    ax_signal.set_title(title)
    fig.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f'Saved overlay plot: {save_path}')

    plt.show()
    return fig, ax_signal, ax_label


# Plot the raw-signal model validation predictions.
# This file is created when the notebook is run with FEATURE_MODE = 'raw_signal'.
raw_signal_val_predictions_csv = output_dir / 'lstm_raw_signal_val_predictions.csv'
raw_signal_overlay_png = output_dir / 'lstm_raw_signal_val_predictions_overlay.png'

if raw_signal_val_predictions_csv.exists():
    plot_validation_prediction_overlay(
        predictions_csv=raw_signal_val_predictions_csv,
        raw_signal=val_eeg,
        true_sample_labels=val_labels,
        fs=fs,
        max_duration_sec=None,      # set to e.g. 30.0 for a shorter zoomed plot
        start_time_sec=0.0,
        save_path=raw_signal_overlay_png,
    )
elif cfg.feature_mode == 'raw_signal' and 'val_predictions_csv' in globals() and Path(val_predictions_csv).exists():
    # Same file, but this branch also works if you changed the output path variable above.
    plot_validation_prediction_overlay(
        predictions_csv=val_predictions_csv,
        raw_signal=val_eeg,
        true_sample_labels=val_labels,
        fs=fs,
        max_duration_sec=None,
        start_time_sec=0.0,
        save_path=raw_signal_overlay_png,
    )
else:
    print(f'Raw-signal prediction CSV not found: {raw_signal_val_predictions_csv}')
    print("Set FEATURE_MODE = 'raw_signal', rerun the training/prediction cells, then rerun this plotting cell.")
